# 10 - Five-Minute Window Analysis

Controlled pilot step for constructing and inspecting incident-specific 5-minute temporal windows from parsed RouteViews observations.

This notebook does not create Normal labels, final 49-feature rows, CAIDA/RPKI features, or train models.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
PILOT = ROOT / 'data' / 'pilot'

windows = pd.read_csv(PILOT / 'five_minute_window_observation_counts_v1.csv')
density = pd.read_csv(PILOT / 'five_minute_window_density_summary_v1.csv')
coverage = pd.read_csv(PILOT / 'five_minute_incident_coverage_v1.csv')

## Analytical Unit

One temporal analytical unit is one incident-specific 5-minute window. Raw prefix-level BGP observations are counted inside each window; they are not individual ML feature rows.

In [ ]:
windows.head(10)

## Window Status Counts

In [ ]:
windows['temporal_status'].value_counts()

## Many Raw Observations to One Window

In [ ]:
windows[['incident_uid', 'window_id', 'raw_observation_count', 'relevant_observation_count', 'announcement_count', 'withdrawal_count']].head(15)

## Density Summary by Incident

In [ ]:
density

## Empty Relevant-Observation Windows

Empty windows are retained. They are not labeled Normal. In outage periods, absence or withdrawal behavior can itself be informative later.

In [ ]:
empty = windows[windows['relevant_observation_count'] == 0]
empty.groupby(['category', 'temporal_status']).size().reset_index(name='empty_windows')

## Incident Coverage

In [ ]:
coverage

## Pilot-Based Scale Estimate

This is a pilot-based estimate, not a final dataset count.

In [ ]:
per_class = density.groupby('category')['five_minute_windows'].agg(['count', 'mean', 'median', 'sum']).reset_index()
per_class['estimated_full_corpus_windows_at_473_incidents'] = (per_class['mean'] * 473).round().astype(int)
per_class

In [ ]:
print('Total pilot windows:', len(windows))
print('Total relevant observations:', int(windows['relevant_observation_count'].sum()))
print('Total raw observations represented:', int(windows['raw_observation_count'].sum()))
print('Empty relevant-observation windows:', int((windows['relevant_observation_count'] == 0).sum()))
print('No Normal labels:', 'NORMAL' not in set(windows['temporal_status']))
print('All incidents visible in active windows:', bool((coverage['observations_visible_during_expected_active_window'] == 'YES').all()))